# From data curation to LoRA fine-tuning

In this research, I examine & experiment with 8 different ways of selecting training data from FIPI, then fine-tune Qwen2.5-7B on 256 pairs selected by each method.

The question I intend to address is whether choosing different training data changes the model's ability to defend against prompt injection under the same training objective and settings. This notebook brings together the configurations, required inputs and commands for each stage.

- [The experiment settings](#The-experiment-settings)
- [What inputs does the experiment use?](#What-inputs-does-the-experiment-use?)
- [Generating responses and selecting data](#Generating-responses-and-selecting-data)
- [Fine-tuning on the selected data](#Fine-tuning-on-the-selected-data)
- [Evaluating the fine-tuned models](#Evaluating-the-fine-tuned-models)


## The experiment settings

I separate data selection from LoRA training, with the dictionary variable `selection_config` containing the classifier and candidate-pool settings, while `training_config` contains the adapter training settings.

There are two validation sets with different purposes: 500 pairs choose each classifier's layer, and 100 pairs choose each LoRA adapter's checkpoint. The printed counts show these alongside the training, candidate and testing sizes.


In [1]:
# compare the full data selection and LoRA training setup
from pathlib import Path
import json
import sys

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'scripts/curation/configs/config_reference_groups.json').exists())
selection_config = json.loads((root / 'scripts/curation/configs/config_qwen7b.json').read_text())
training_config = json.loads((root / 'scripts/curation/configs/config_reference_groups.json').read_text())

# use separate validation pairs to choose classifier layers and LoRA checkpoints
print('Probe training pairs:', selection_config['selector_training_pairs'])
print('Layer validation pairs:', selection_config['validation_pairs'])
print('Candidate pairs:', selection_config['candidate_pairs'])
print('Pairs selected by each method:', selection_config['selected_pairs'])
print('LoRA validation pairs:', training_config['validation_pairs'])
print('FIPI testing pairs:', training_config['testing_pairs'])
print('LoRA objective:', training_config['objective'])


Probe training pairs: 1500
Layer validation pairs: 500
Candidate pairs: 10000
Pairs selected by each method: 256
LoRA validation pairs: 100
FIPI testing pairs: 200
LoRA objective: Mean token cross-entropy against the existing safe-prompt response under the paired injection prompt, including EOS


## What inputs does the experiment use?

The experiment uses FIPI training and testing records, the Qwen tokenizer and two fitted Jacobian lenses, listed in the dictionary variable `inputs`.

The loop checks which files are available, while repeating the original lens comparison requires the same fitted matrices or their original fitting prompts and settings.


In [2]:
inputs = {
    'FIPI training': selection_config['training_data'],
    'FIPI testing': selection_config['testing_data'],
    'Qwen tokenizer': selection_config['model'] + '/tokenizer.json',
    **{method: details['path'] for method, details in selection_config['lenses'].items()},
}

# exact reproduction needs both original lenses, including the one fitted on private source prompts
for name, relative_path in inputs.items():
    status = 'present' if (root / relative_path).is_file() else 'missing'
    print(f'{name}: {status} | {relative_path}')


FIPI training: present | data/FIPI/raw/train_100000.json
FIPI testing: present | data/FIPI/raw/test_10000.json
Qwen tokenizer: present | model/Qwen2.5-7B-Instruct/tokenizer.json
jlens_llmail: present | model/artifacts/qwen2.5-7b/llmail-v1/lens.pt
jlens_synthetic: present | model/artifacts/qwen2.5-7b/old-classifier-titlecase-v1/lens.pt


## Generating responses and selecting data

I first divide related source prompts into separate groups, then collect hidden state vectors for probe training and validation before choosing each classifier's layer.

After that, the classifiers score the candidate prompts, the base model generates both responses for each pair, and each method selects 256 eligible pairs. The list variable `commands` follows this order and prints the execution commands for a separately approved run.


In [3]:
# select the Qwen2.5-7B configuration explicitly before importing the selection modules
prefix = 'FIPI_CONFIG=scripts/curation/configs/config_qwen7b.json FIPI_SKIP_NOTEBOOKS=1 python -m '
commands = [
    prefix + 'scripts.curation.preparation.pool',
    prefix + 'scripts.curation.preparation.Record_hidden_state_vectors_and_model_response --stage probe_states',
    prefix + 'scripts.curation.preparation.preparation --stage fit',
    prefix + 'scripts.curation.preparation.Record_hidden_state_vectors_and_model_response --stage candidate_states',
    prefix + 'scripts.curation.preparation.preparation --stage score',
    prefix + 'scripts.curation.preparation.Record_hidden_state_vectors_and_model_response --stage responses',
    prefix + 'scripts.curation.preparation.preparation --stage select',
]

# print commands only so Run All cannot start probe fitting or language-model generation
for command in commands:
    print(command)


FIPI_CONFIG=scripts/curation/configs/config_qwen7b.json FIPI_SKIP_NOTEBOOKS=1 python -m scripts.curation.preparation.pool
FIPI_CONFIG=scripts/curation/configs/config_qwen7b.json FIPI_SKIP_NOTEBOOKS=1 python -m scripts.curation.preparation.Record_hidden_state_vectors_and_model_response --stage probe_states
FIPI_CONFIG=scripts/curation/configs/config_qwen7b.json FIPI_SKIP_NOTEBOOKS=1 python -m scripts.curation.preparation.preparation --stage fit
FIPI_CONFIG=scripts/curation/configs/config_qwen7b.json FIPI_SKIP_NOTEBOOKS=1 python -m scripts.curation.preparation.Record_hidden_state_vectors_and_model_response --stage candidate_states
FIPI_CONFIG=scripts/curation/configs/config_qwen7b.json FIPI_SKIP_NOTEBOOKS=1 python -m scripts.curation.preparation.preparation --stage score
FIPI_CONFIG=scripts/curation/configs/config_qwen7b.json FIPI_SKIP_NOTEBOOKS=1 python -m scripts.curation.preparation.Record_hidden_state_vectors_and_model_response --stage responses
FIPI_CONFIG=scripts/curation/configs/c

## Fine-tuning on the selected data

Each selected data group trains a separate LoRA adapter from the same base model, using the injection prompt as input and its safe counterpart's generated response as the reference answer.

The dictionary variable `training_config` contains the shared settings, and the printed commands identify which data group each adapter uses. All 5 epochs run before validation loss determines the checkpoint used for testing.


In [4]:
# print the settings together so a new run can be reviewed before it starts
fields = ['model', 'model_revision', 'training_pairs', 'validation_pairs', 'testing_pairs', 'objective', 'layers', 'target_modules', 'epochs', 'rank', 'alpha', 'dropout', 'learning_rate', 'schedule', 'pairs_per_update', 'gradient_accumulation', 'max_input_tokens', 'max_task_response_tokens', 'evaluation_max_new_tokens', 'seed', 'compute_dtype']
for field in fields:
    print(f'{field}: {training_config[field]}')

# the index chooses one of the eight data groups, not a different training objective
for index, method in enumerate(training_config['methods']):
    print(method, f'python -m scripts.curation.training.train_reference_groups --method-index {index}')


model: model/Qwen2.5-7B-Instruct
model_revision: a09a35458c702b33eeacc393d103063234e8bc28
training_pairs: 256
validation_pairs: 100
testing_pairs: 200
objective: Mean token cross-entropy against the existing safe-prompt response under the paired injection prompt, including EOS
layers: [10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
target_modules: ['q_proj', 'v_proj', 'up_proj', 'down_proj']
epochs: 5
rank: 16
alpha: 32
dropout: 0.05
learning_rate: 5e-05
schedule: Linear warmup followed by cosine decay to 0.000005, matching the FIPI training schedule
pairs_per_update: 1
gradient_accumulation: 1
max_input_tokens: 4096
max_task_response_tokens: 2048
evaluation_max_new_tokens: 1024
seed: 42
compute_dtype: bfloat16
random python -m scripts.curation.training.train_reference_groups --method-index 0
text_diversity python -m scripts.curation.training.train_reference_groups --method-index 1
model_failures python -m scripts.curation.training.train_reference_groups --method-index 2
logistic_regression p

## Evaluating the fine-tuned models

I evaluate the base model and all 8 adapters using each benchmark's tasks and attack checks, with the dictionary variable `evaluators` identifying the corresponding scripts.

FIPI's benchmark checks for the requested injected answer, LLMail-Inject checks message-sending attempts under an adapted summarization task, and the simulated agent tasks check actions against their injection goals. The evaluation notebook calculates ASR (Attack Success Rate).


In [5]:
evaluators = {
    'FIPI': 'scripts/curation/training/train_reference_groups.py',
    'LLMail': 'scripts/curation/evaluation/evaluate_llmail_reference.py',
    'Synthetic business': 'scripts/curation/evaluation/evaluate_synthetic_business.py',
    'BIPIA': 'scripts/curation/evaluation/evaluate_bipia_reference.py',
    'AgentDojo, AgentDyn, InjecAgent': 'scripts/curation/evaluation/evaluate_task_benchmarks.py',
    'AgentLAB': 'scripts/curation/evaluation/evaluate_agentlab_reference.py',
    'Corrected multi-turn evaluation': 'scripts/curation/evaluation/repair_task_evaluations.py',
}

# identify the evaluator used for each benchmark and its multi-turn continuation
for benchmark, script in evaluators.items():
    print(f'{benchmark}: {script}')


FIPI: scripts/curation/training/train_reference_groups.py
LLMail: scripts/curation/evaluation/evaluate_llmail_reference.py
Synthetic business: scripts/curation/evaluation/evaluate_synthetic_business.py
BIPIA: scripts/curation/evaluation/evaluate_bipia_reference.py
AgentDojo, AgentDyn, InjecAgent: scripts/curation/evaluation/evaluate_task_benchmarks.py
AgentLAB: scripts/curation/evaluation/evaluate_agentlab_reference.py
Corrected multi-turn evaluation: scripts/curation/evaluation/repair_task_evaluations.py
